In [ ]:
from approx.approximate import ApprOXimate
from approx.feature_engineering import MaterialFeatureExtractor
from mendeleev.fetch import fetch_table
import pandas as pd

# Clean Dataset

some compositions will not work very well with ApprOxiMate as it's mainly for inorganic compounds that need to be charge balanced

In [ ]:
icsd_df = pd.read_csv("ICSD_CrystStrucData.csv")
icsd_df

,HMS,StructuredFormula,Temperature,Pressure,crystal_system
0,P 4/m m m,Nd1Ba1Mn1Fe1O5.45,293.0,0.101325,tetragonal
1,P 4/m m m,Nd1Ba1Mn1Fe1O5.17,293.0,0.101325,tetragonal
2,P 4/m m m,Nd1Ba1Mn1Fe1O5.09,293.0,0.101325,tetragonal
3,I 4/m m m,Sr2Mn2.275Cr0.725As2O2,300.0,0.101325,tetragonal
4,P 42 m c,Ca1Mn1Ti1.8V0.2O6,293.0,0.101325,tetragonal
...,...,...,...,...,...
2296,F d -3 Z,Mn21.5Rb49Al92Si100O384,294.0,0.101325,cubic
2297,F 4 3 2,Ca6.3Mn3Ga4.4Al1.3O18,293.0,0.101325,cubic
2298,F -4 3 m,Li1In1Cr3.8Mn0.2O8,293.0,0.101325,cubic
2299,F -4 3 m,Li1In1Cr3.6Mn0.4O8,293.0,0.101325,cubic


In [ ]:
approx = ApprOXimate()

for formula in icsd_df['StructuredFormula'].tolist():
    try:
        formula_dict = approx.charge_balance(formula, return_format='string')
    except:
        print(formula)

Ca8Mn10.64Si12.32O56H18
Mn2H46O101Si2W24
La6H113O132Mn2V25
Ce6H109O130Mn2V25
Pr6H113O132Mn2V25
Mn6H88O110W19Zn3
H76Mn5.5O102W18.5Sb2
Na10Mn5H148O156W24
Na12Mn1Nb12O88H100
K2.16Mn16Si26.9O75.8H8
K3.68Mn15.904Si25.472O80.4H23.424
Na12Mn1Nb12O90H104
Mn16Si12As3O57H17
Mn28Cs36Al92Si100O384
Mn28.5Si135Al57O411.2H54.4
Mn21.5Rb49Al92Si100O384


In [5]:
def charge_balance_ok(formula):
    try:
        approx.charge_balance(formula, return_format='string')
        return True
    except Exception:
        return False

mask = icsd_df['StructuredFormula'].apply(charge_balance_ok)
icsd_df = icsd_df[mask].reset_index(drop=True)

these formulas can't be charge balanced therefore they are removed from the dataset. This is because some of the values for an element is very large

In [ ]:
from tqdm import tqdm

def featurize_df(df, extractor, formula_col="Compound"):
    formulas = df[formula_col].tolist()
    feature_rows = extractor.featurize_many(formulas)
    feat_df = pd.DataFrame(feature_rows, index=df.index)

    # Keep only rows that featurized successfully
    feature_cols = feat_df.drop(columns=["formula"], errors="ignore")
    if feature_cols.shape[1] == 0:
        return df.iloc[0:0].copy()

    valid_rows = feature_cols.notna().any(axis=1)
    df_valid = df.loc[valid_rows]
    feat_valid = feat_df.loc[valid_rows]

    failed_formulas = df.loc[~valid_rows, formula_col]
    for formula in failed_formulas:
        print("Formula Fail:", formula)

    return df_valid.join(feat_valid)

# Initialize modules
approx = ApprOXimate()
ptable = fetch_table("elements")
extractor = MaterialFeatureExtractor(approx, ptable, mode="both")

# Run featurization
df_feat = featurize_df(icsd_df, extractor, formula_col="StructuredFormula")
df_feat.head()

Error in Mn1Si1F6D12O6: 'D'
Error in Na0.6Mn2O5.5D2H1: 'D'
Formula Fail: Mn1Si1F6D12O6
Formula Fail: Na0.6Mn2O5.5D2H1


,HMS,StructuredFormula,Temperature,Pressure,crystal_system,formula,all_valence_s_sum,all_valence_s_avg,all_valence_s_dev,all_valence_s_min,...,all_gordy_en_max,all_gordy_en_range,all_gordy_en_mode,all_mb_en_sum,all_mb_en_avg,all_mb_en_dev,all_mb_en_min,all_mb_en_max,all_mb_en_range,all_mb_en_mode
0,P 4/m m m,Nd1Ba1Mn1Fe1O5.45,293.0,0.101325,tetragonal,Nd1Ba1Mn1Fe1O5.45,10.90,1.153439,0.988158,0.0,...,0.465308,0.424263,0.041044,0.460741,0.048756,0.025544,0.0,0.063694,0.063694,0.063492
1,P 4/m m m,Nd1Ba1Mn1Fe1O5.17,293.0,0.101325,tetragonal,Nd1Ba1Mn1Fe1O5.17,10.34,1.127590,0.991827,0.0,...,0.465308,0.424263,0.041044,0.443294,0.048342,0.025803,0.0,0.063694,0.063694,0.063492
2,P 4/m m m,Nd1Ba1Mn1Fe1O5.09,293.0,0.101325,tetragonal,Nd1Ba1Mn1Fe1O5.09,10.18,1.119912,0.992785,0.0,...,0.465308,0.424263,0.041044,0.438309,0.048219,0.025878,0.0,0.063694,0.063694,0.063492
3,I 4/m m m,Sr2Mn2.275Cr0.725As2O2,300.0,0.101325,tetragonal,Sr2Mn2.275Cr0.725As2O2,8.00,0.380952,0.785353,0.0,...,0.378548,0.337504,0.203202,1.119075,0.053289,0.039683,0.0,0.166667,0.166667,0.042553
4,P 42 m c,Ca1Mn1Ti1.8V0.2O6,293.0,0.101325,tetragonal,Ca1Mn1Ti1.8V0.2O6,12.00,1.200000,0.979796,0.0,...,0.298507,0.257463,0.041044,0.450422,0.045042,0.025328,0.0,0.063492,0.063492,0.063492


In [20]:
df_feat.to_csv('ICSD_featurised_data.csv', index=False)